# 02. Тестирование устойчивости и скорости inference

Использует сохранённую модель из настоящего проекта; исходный датасет не нужен. Это **диагностические** тесты, не измерение обобщающей точности. Проверяются стабильность предсказаний, Unicode, шум, вариации регистра и замеры времени модели без HTTP/БД.

In [ ]:
import os, json, time
from pathlib import Path
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
ROOT = Path.cwd().resolve()
if not (ROOT/'artifacts').exists(): ROOT = ROOT.parent
MODEL_PATH = Path(os.getenv('MODEL_PATH', str(ROOT/'artifacts/baseline_model.joblib')))
model = joblib.load(MODEL_PATH)  # Только доверенный артефакт команды
classes=set(model.classes_)
assert classes=={'neutral','offensive','toxic'}
print('Модель загружена:',MODEL_PATH.name)

## Диагностика вводов: без предположений о правильном классе

In [ ]:
cases = [
    ('нейтральное_сообщение','Спасибо за помощь!'),
    ('агрессия','Отстань от меня, пожалуйста!'),
    ('угроза','Я тебе угрожаю'),
    ('emoji','😀🔥👍'),
    ('смешанные_языки','Спасибо, friend!'),
    ('транслит','privet kak dela'),
    ('опечатка','спосибо за памощь'),
    ('символы','!!!????'),
    ('перенос_строки','Привет\nСпасибо'),
    ('длинный_текст','сообщение ' * 400),
]
labels=[name for name,_ in cases]
texts=[text for _,text in cases]
pred=model.predict(texts)
prob=model.predict_proba(texts)
assert np.isfinite(prob).all() and (prob>=0).all() and np.allclose(prob.sum(axis=1),1,atol=1e-6)
display(pd.DataFrame({'case':labels,'predicted':pred,'confidence':prob.max(axis=1)}))
print('Все вероятности конечны и нормализованы')

## Детерминированность и простые переформулировки

In [ ]:
assert np.array_equal(model.predict(texts), pred)
assert np.allclose(model.predict_proba(texts), prob)
pairs=[('Спасибо за помощь','СПАСИБО ЗА ПОМОЩЬ'),
       ('Спасибо за помощь','Спасибо  за    помощь'),
       ('Спасибо за помощь','Спасибо за помощь!!!'),
       ('Мне это не нравится','Мне это не нравится!!!')]
base=model.predict([x for x,_ in pairs]); changed=model.predict([x for _,x in pairs])
shifts=pd.DataFrame({'variant':['case','spaces','punctuation','negative_punctuation'],
                    'original':base,'modified':changed,'prediction_changed':base!=changed})
display(shifts)
print('Смена предсказания при возмущении — сигнал для анализа, а не автоматическая ошибка')

## Время локального инференса (без FastAPI, сети и PostgreSQL)

In [ ]:
_ = model.predict(['тестовый прогрев'])
results=[]
for i in range(50):
    start=time.perf_counter()
    model.predict(['Спасибо за помощь'])
    results.append((time.perf_counter()-start)*1000)
print({'n':len(results),'p50_ms':round(float(np.percentile(results,50)),2),
       'p95_ms':round(float(np.percentile(results,95)),2),'p99_ms':round(float(np.percentile(results,99)),2)})
plt.hist(results,bins=10)
plt.xlabel('Время модели, мс')
plt.ylabel('Количество вызовов')
plt.show()

## Ограничения

Данный ноутбук не доказывает качество на новых доменах и не заменяет нагрузочное тестирование реального HTTP API. Валидация пользовательского текста — отдельный слой в `app/main.py`. Для HTTP-нагрузки проверяйте `/predict` внешним инструментом; Prometheus показывает реальные задержки и частоты ошибок. Не сохраняйте сырые пользовательские тексты в метрики/логи.